# GraphTrustBench downstream readout
CPU preparation is verified separately from optional model execution. Ordered top-five lists may differ in membership or order. This is a selected changed-context diagnostic, not an estimate over every task.
Run with DRY_RUN=True first. Missing source fails closed; no model imports, serving or calls occur in dry run. Before live use verify model availability, revision, quota and harness compatibility. Model serving and repair performance remain unverified.


In [ ]:
# --- cell 1: configuration -------------------------------------------------------
import json, os, re, sys, shutil
from pathlib import Path

DRY_RUN = True                     # set False only after the dry run looks right
MODEL_NAME = 'gemma-4-e2b-it-qat-w4a16-ct'   # smallest competition model: cheapest read-out
DATA_DIR = Path('/kaggle/input/competitions/gemma-4-developer-agent')
WORKING_DIR = Path('/kaggle/working')
REPO_DIR = Path('/kaggle/input/graphtrustbench')     # the public release checkout
RUN_DIR = REPO_DIR / 'paper' / 'results' / '2026-11-07_heldout_v3'
RESULT_PATH = WORKING_DIR / 'agent_readout_results.txt'
sys.path.insert(0, str(REPO_DIR / 'paper' / 'scripts'))

WORKING_DIR.mkdir(parents=True, exist_ok=True)
for label, path, required in (('data dir', DATA_DIR, True), ('repo dir', REPO_DIR, True),
                               ('run dir', RUN_DIR, True)):
    print(f'{label:9}: {path} exists={path.exists()}')
    assert not required or path.exists(), f'{label} missing: {path}'
print('model     :', MODEL_NAME)
print('dry run   :', DRY_RUN)

In [ ]:
# Validate the frozen experiment before preparing any downstream context.
from render_pilot_report import load_verified_run
from agent_readout import paired_contexts, prepare_contexts
PAPER = REPO_DIR / 'paper'
audit, selection, run_manifest, mappings, rows, run_summary = load_verified_run(
    PAPER/'reports/competition_audit_full_2026-11-06.json', RUN_DIR,
    PAPER/'results/2026-11-07_heldout_mapping', PAPER/'data/heldout_2026-11-08.json',
    PAPER/'config/pilot_v3.json')
pairs = paired_contexts(rows, top_k=5, corruption='spurious', severity=0.4, seed=13)
assert pairs, 'No eligible paired contexts'
print(len(pairs), 'pairs with changed ordered top-five files')


In [ ]:
# Missing task/source or changed snapshot aborts before inference.
records = prepare_contexts(DATA_DIR, selection, audit, pairs)
assert len(records) == 2 * len(pairs)
print(len(records), 'verified task-specific prompts; no source text printed')


In [ ]:
# Record hashes and lengths only; saved notebook outputs must not expose corpus text.
for record in records:
    print(record['instance_id'], record['condition'], len(record['prompt']), record['prompt_sha256'])


In [ ]:
# --- cell 5: DRY RUN - verify the prompts before spending GPU time ----------------
if DRY_RUN:
    lengths = [len(r['prompt']) for r in records]
    print('prompt length: min', min(lengths), 'max', max(lengths), 'mean', sum(lengths)//len(lengths))
    short = [r for r in records if len(r['prompt']) < 200]
    print('suspiciously short prompts:', len(short))
    for r in short[:5]:
        print('   ', r['instance_id'], r['condition'], repr(r['prompt'][:200]))
    print('DRY RUN COMPLETE - inspect the prompt above, then set DRY_RUN = False in cell 1.')
else:
    print('live run')

## Optional model execution
The serving adapter requires validation against the installed competition harness before live use. CPU smoke tests do not verify serving compatibility.


In [ ]:
if DRY_RUN:
    print('DRY RUN: model imports and serving skipped')
else:
    # --- cell 6: start the vLLM server (official pattern) ----------------------------
    import litellm, torch
    from adk_submission import VllmConfig, VllmServer
    
    litellm.drop_params = True
    INFERENCE_API_KEY = os.environ.get('LOCAL_API_KEY', 'EMPTY')
    MODEL_PATH = Path(f'/kaggle/input/models/google/gemma-4/other/{MODEL_NAME}/2')
    assert MODEL_PATH.exists(), f'Model weights not found at {MODEL_PATH}; attach the model input.'
    
    MODEL_VLLM_EXTRA_ARGS = {
        'gemma-4-e2b-it-qat-w4a16-ct': (
            '--hf-overrides',
            '{"architectures": ["Gemma4ForCausalLM"], "model_type": "gemma4"}',
        ),
    }
    gpu_count = torch.cuda.device_count() if torch.cuda.is_available() else 1
    tp_size = 4 if gpu_count >= 4 else (2 if gpu_count >= 2 else 1)
    print('gpus:', gpu_count, 'tensor parallel:', tp_size)
    
    vllm_config = VllmConfig(
        model=str(MODEL_PATH),
        tensor_parallel_size=tp_size,
        extra_args=list(MODEL_VLLM_EXTRA_ARGS.get(MODEL_NAME, ())),
    )
    server_instance = VllmServer(config=vllm_config)
    server_instance.start()
    base_url = server_instance.base_url
    print('vLLM base url:', base_url)


In [ ]:
# --- cell 7: run the paired calls (greedy, fixed prompt) --------------------------
def ask(prompt):
    response = litellm.completion(
        model=MODEL_NAME,
        messages=[{'role': 'user', 'content': prompt}],
        temperature=0.0,
        max_tokens=32,
        api_key=INFERENCE_API_KEY,
        base_url=base_url,
    )
    return response.choices[0].message.content

answers = {}
if DRY_RUN:
    print('DRY RUN - no model calls made')
else:
    for index, record in enumerate(records):
        try:
            answer = ask(record['prompt'])
        except Exception as exc:          # one failed call must not lose the run
            print('call failed', record['instance_id'], record['condition'], exc)
            answer = ''
        answers[(record['instance_id'], record['condition'])] = answer
        print(f"[{index+1}/{len(records)}] {record['instance_id']} {record['condition']}: {answer!r}")

In [ ]:
# --- cell 8: score with the unit-tested scorer and save ---------------------------
from agent_readout import score_answers

summary = ({'status': 'cpu_preparation_only', 'prompts': len(records)} if DRY_RUN else score_answers(records, answers))
summary['model'] = MODEL_NAME
summary['dry_run'] = DRY_RUN
summary['corruption'] = 'spurious'
summary['severity'] = 0.4
summary['seed'] = 13
summary['input_provenance'] = {'run_manifest': run_manifest, 'prompt_hashes': [r['prompt_sha256'] for r in records]}
summary['source_hashes'] = {p.name: __import__('hashlib').sha256(p.read_bytes()).hexdigest() for p in (PAPER/'scripts').glob('*.py')}
summary['raw_answers'] = {f'{k[0]}|{k[1]}': v for k, v in answers.items()}
RESULT_PATH.write_text(json.dumps(summary, indent=2, sort_keys=True) + '\n')

for arm in (() if DRY_RUN else ('clean', 'corrupted')):
    stats = summary['arms'][arm]
    print(f"{arm:10} accuracy={stats['accuracy']} faithfulness={stats['faithfulness']} "
          f"answerable={stats['answerable']} unanswerable={stats['unanswerable']}")
if not DRY_RUN: print('discordant:', summary['discordant'])
print('saved ->', RESULT_PATH)
if not DRY_RUN:
    print('Copy the result file into the release repository and commit it with the paper artifacts.')